In [9]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,models,callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from sklearn.metrics import ( confusion_matrix,
    classification_report,
    roc_curve,auc)
print(f"   TensorFlow version : {tf.__version__}")
print(f"   Keras version      : {keras.__version__}")
print()
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU detected: {gpus}")
else:
    print("No GPU detected — using CPU")
tf.random.set_seed(42)
np.random.seed(42)
print()
print("Random seed set to 42 — results will be reproducible")

   TensorFlow version : 2.21.0
   Keras version      : 3.14.0

No GPU detected — using CPU

Random seed set to 42 — results will be reproducible


In [7]:
# ── Paths ─────────────────────────────────────────────────────────────────
PROCESSED_ROOT = Path("../data/processed")
TRAIN_DIR      = PROCESSED_ROOT / "training"
VAL_DIR        = PROCESSED_ROOT / "validation"
TEST_DIR       = PROCESSED_ROOT / "testing"
MODELS_DIR     = Path("../models")
RESULTS_DIR    = Path("../results")

# Create directories 
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────
IMG_SIZE      = (128, 128)   
IMG_SHAPE     = (128, 128, 3) 
BATCH_SIZE    = 32            # Files processed together in one training step
EPOCHS        = 30            # Maximum training rounds
LEARNING_RATE = 0.001         # How fast the model adjusts its weights
DROPOUT_RATE  = 0.3           # Fraction of neurons randomly disabled each step

# Model save path
MODEL_SAVE_PATH = MODELS_DIR / "vocal_armor_best.keras"

# ── Verify processed data exists ──────────────────────────────────────────
print("Checking processed spectrogram folders...")
print()

all_good = True
for split in ["training", "validation", "testing"]:
    for label in ["real", "fake"]:
        folder = PROCESSED_ROOT / split / label
        count  = len(list(folder.glob("*.png"))) if folder.exists() else 0
        status = "✅" if count > 0 else "❌"
        print(f"  {status}  processed/{split}/{label:<12} → {count:>7,} images")
        if count == 0:
            all_good = False

print()
if all_good:
    print("All folders found! Ready to train.")
else:
    print("Some folders are empty. Make sure Cell 12 in notebook 01 completed fully.")

print()
print("Hyperparameters:")
print(f"Image size     : {IMG_SIZE}")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Max epochs     : {EPOCHS}")
print(f"Learning rate  : {LEARNING_RATE}")
print(f"Dropout rate   : {DROPOUT_RATE}")

Checking processed spectrogram folders...

  ✅  processed/training/real         →   6,978 images
  ✅  processed/training/fake         →   6,978 images
  ✅  processed/validation/real         →   1,413 images
  ✅  processed/validation/fake         →   1,413 images
  ✅  processed/testing/real         →     544 images
  ✅  processed/testing/fake         →     544 images

All folders found! Ready to train.

Hyperparameters:
Image size     : (128, 128)
Batch size     : 32
Max epochs     : 30
Learning rate  : 0.001
Dropout rate   : 0.3
